# MLflow on RHOAI — Live Demo

**~15 minutes** | Auto-trace LLM calls, compare prompt strategies, observe agentic workflows.

> *"Every LLM call in your AI application is automatically traced — no code changes required."*

## Setup

In [ ]:
!pip install -q mlflow openai

In [ ]:
import os

# --- EDIT THIS: paste your MaaS API key ---
os.environ["MAAS_API_KEY"] = "<YOUR_MAAS_API_KEY>"

# Cluster endpoints (no changes needed if running inside the cluster)
os.environ["MAAS_URL"] = "https://maas.apps.ocp.xlwsd.sandbox1213.opentlc.com/rhoai-playground/qwen3-8b/v1"

# Use the external route — the inline trace widget renders in the browser,
# which cannot resolve internal .svc addresses
os.environ["MLFLOW_TRACKING_URI"] = "https://rh-ai.apps.ocp.xlwsd.sandbox1213.opentlc.com/mlflow/"
os.environ["MLFLOW_TRACKING_TOKEN"] = os.popen("cat /var/run/secrets/kubernetes.io/serviceaccount/token").read().strip()

---
## Part 1: LLM Tracing (5 min) — The Hook

One line of code (`mlflow.openai.autolog()`) captures every LLM call automatically.

In [ ]:
import mlflow
import openai
import time

mlflow.openai.autolog()

mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
mlflow.set_workspace("rhoai-playground")
mlflow.set_experiment("llm-tracing-demo")

client = openai.OpenAI(
    base_url=os.environ["MAAS_URL"],
    api_key=os.environ["MAAS_API_KEY"],
)

prompts = [
    "Explain Kubernetes to a 5-year-old in two sentences.",
    "What are the three biggest risks of running ML models in production?",
    "Write a haiku about container orchestration.",
]

for prompt in prompts:
    print(f"\n--- Prompt: {prompt}")
    response = client.chat.completions.create(
        model="qwen3-8b",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=256,
    )
    print(response.choices[0].message.content[:200])

mlflow.flush_trace_async_logging(terminate=True)
time.sleep(3)
print("\nDone. Check MLflow UI -> Traces tab.")

### What to check in the UI

1. Open **MLflow** (Develop & train > Experiments in the RHOAI dashboard)
2. Select workspace **rhoai-playground** (top-left dropdown)
3. Click the **Traces** tab
4. Each LLM call appears as a trace — click one to expand
5. Note: full request/response body, latency, token counts, model name — all captured automatically

> **Talking point**: *"This is `mlflow.openai.autolog()` — one line of code. Works with any OpenAI-compatible endpoint, which is exactly what vLLM on RHOAI exposes. No SDK lock-in."*

---
## Part 2: Experiment Tracking — Prompt Comparison (5 min)

Compare three prompt strategies for the same task. Each run logs parameters, metrics, and the full response.

In [ ]:
import mlflow
import openai
import time

mlflow.openai.autolog()

mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
mlflow.set_workspace("rhoai-playground")
mlflow.set_experiment("prompt-engineering")

client = openai.OpenAI(
    base_url=os.environ["MAAS_URL"],
    api_key=os.environ["MAAS_API_KEY"],
)

TASK = "Summarize the benefits of Red Hat OpenShift for enterprise AI workloads."

strategies = [
    {
        "name": "direct",
        "system": "You are a helpful assistant.",
        "temperature": 0.3,
        "max_tokens": 200,
    },
    {
        "name": "concise-expert",
        "system": "You are a senior cloud architect. Be concise. Use bullet points.",
        "temperature": 0.1,
        "max_tokens": 200,
    },
    {
        "name": "creative",
        "system": "You are a tech evangelist writing for a blog audience. Be engaging.",
        "temperature": 0.9,
        "max_tokens": 300,
    },
]

for strategy in strategies:
    with mlflow.start_run(run_name=strategy["name"]):
        mlflow.log_params({
            "strategy": strategy["name"],
            "system_prompt": strategy["system"][:80],
            "temperature": strategy["temperature"],
            "max_tokens": strategy["max_tokens"],
            "model": "qwen3-8b",
        })

        response = client.chat.completions.create(
            model="qwen3-8b",
            messages=[
                {"role": "system", "content": strategy["system"]},
                {"role": "user", "content": TASK},
            ],
            temperature=strategy["temperature"],
            max_tokens=strategy["max_tokens"],
        )

        output = response.choices[0].message.content
        usage = response.usage

        mlflow.log_metrics({
            "prompt_tokens": usage.prompt_tokens,
            "completion_tokens": usage.completion_tokens,
            "total_tokens": usage.total_tokens,
            "response_length": len(output),
        })
        mlflow.log_text(output, "response.txt")

        print(f"\n[{strategy['name']}] tokens={usage.total_tokens} len={len(output)}")
        print(output[:150] + "...")

mlflow.flush_trace_async_logging(terminate=True)
time.sleep(3)
print("\nDone. Compare runs in MLflow UI.")

### What to check in the UI

1. Click **Experiments** > **prompt-engineering**
2. Three runs: `direct`, `concise-expert`, `creative`
3. Select all three > click **Compare**
4. Compare **Parameters** (system prompt, temperature) and **Metrics** (token usage, response length)
5. Click a run > **Artifacts** > `response.txt` to see actual output

> **Talking point**: *"This is how ML teams do prompt engineering at scale. Every variation is versioned, measured, and comparable. No more losing track of what you tried."*

---
## Part 3: Agentic Workflow Tracing (5 min)

MLflow captures multi-step tool-calling workflows — not just single completions.

In [ ]:
import mlflow
import openai
import json
import time

mlflow.openai.autolog()

mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
mlflow.set_workspace("rhoai-playground")
mlflow.set_experiment("agent-tracing")

client = openai.OpenAI(
    base_url=os.environ["MAAS_URL"],
    api_key=os.environ["MAAS_API_KEY"],
)

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_cluster_status",
            "description": "Get the current status of an OpenShift cluster",
            "parameters": {
                "type": "object",
                "properties": {
                    "cluster_name": {"type": "string", "description": "Name of the cluster"},
                },
                "required": ["cluster_name"],
            },
        },
    },
]

def fake_get_cluster_status(cluster_name):
    return json.dumps({
        "cluster": cluster_name,
        "status": "healthy",
        "nodes": 6,
        "gpu_nodes": 2,
        "running_models": ["qwen3-8b"],
        "cpu_utilization": "42%",
    })

with mlflow.start_run(run_name="ops-agent"):
    messages = [
        {"role": "system", "content": "You are an OpenShift operations assistant."},
        {"role": "user", "content": "Check the status of cluster 'production-east' and tell me if we have capacity for another model."},
    ]

    # Step 1: model decides to call a tool
    response = client.chat.completions.create(
        model="qwen3-8b", messages=messages, tools=tools, max_tokens=512,
    )
    msg = response.choices[0].message

    if msg.tool_calls:
        messages.append(msg)
        for tc in msg.tool_calls:
            result = fake_get_cluster_status(
                json.loads(tc.function.arguments).get("cluster_name", "unknown")
            )
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
            print(f"Tool call: {tc.function.name}({tc.function.arguments}) -> {result}")

        # Step 2: model synthesizes the tool result
        response = client.chat.completions.create(
            model="qwen3-8b", messages=messages, max_tokens=512,
        )
        print(f"\nAgent response:\n{response.choices[0].message.content}")
    else:
        print(f"Response (no tool call):\n{msg.content}")

mlflow.flush_trace_async_logging(terminate=True)
time.sleep(3)
print("\nDone. Check Traces tab for multi-step agent trace.")

### What to check in the UI

1. Go to **Traces** in the `agent-tracing` experiment
2. The trace shows the multi-step flow: initial call > tool invocation > follow-up call
3. Each step has its own span with inputs/outputs

> **Talking point**: *"As you build agentic AI apps — chains of LLM calls with tool use — MLflow captures the entire execution graph. Debug exactly where an agent went wrong."*

---
## Key Messages

| Concern | Answer |
|---------|--------|
| "We already use W&B" | MLflow is open-source, runs on-prem, no data leaves your cluster |
| "How much code change?" | One line: `mlflow.openai.autolog()` |
| "Works with our models?" | Any OpenAI-compatible endpoint — vLLM, TGI, or external |
| "Production ready?" | Same instance for dev experiments and prod traces. RBAC via OpenShift |
| "Cost?" | Included with RHOAI — no per-seat licensing |